In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 — Silver Transformation
# MAGIC **Layer:** Silver | Deterministic DQ flags, type casting, quarantine routing
# MAGIC
# MAGIC - 6 boolean DQ flags per record + composite `dq_passed`
# MAGIC - Failed records routed to `quarantine_bronze`
# MAGIC - Change Data Feed enabled for incremental Gold reads

# COMMAND ----------

from pyspark.sql.functions import (
    col, trim, upper, when, isnull,
    current_timestamp, lit, regexp_extract,
    regexp_replace, expr
)
from pyspark.sql.types import DecimalType

CATALOG = "fintech_lakehouse_dev"
SCHEMA  = "transactions"
BRONZE_CORE  = f"{CATALOG}.{SCHEMA}.bronze_core_banking"
BRONZE_CARD  = f"{CATALOG}.{SCHEMA}.bronze_card_auth"
SILVER_CORE  = f"{CATALOG}.{SCHEMA}.silver_core_banking"
SILVER_CARD  = f"{CATALOG}.{SCHEMA}.silver_card_auth"
QUARANTINE   = f"{CATALOG}.{SCHEMA}.quarantine_bronze"

VALID_CURRENCIES = ["USD", "EUR", "GBP", "CAD", "MXN"]
VALID_TXN_TYPES  = ["PURCHASE", "TRANSFER", "REFUND", "WITHDRAWAL"]
VALID_STATUSES   = ["APPROVED", "PENDING", "DECLINED"]

In [0]:
# COMMAND ----------
# Load bronze tables (dq_passed only)
core_df = spark.table(BRONZE_CORE)
card_df = spark.table(BRONZE_CARD)

print(f"Bronze core rows: {core_df.count()}")
print(f"Bronze card rows: {card_df.count()}")

In [0]:
# COMMAND ----------
# Profile actual values before DQ — run once to confirm valid value lists
print("=== transaction_type values ===")
core_df.groupBy("transaction_type").count().orderBy("count", ascending=False).show()

print("=== status values ===")
core_df.groupBy("status").count().orderBy("count", ascending=False).show()

print("=== currency values ===")
core_df.groupBy("currency").count().orderBy("count", ascending=False).show()

In [0]:
# COMMAND ----------
def apply_silver_transforms(df, source_table_name: str):
    """
    Apply type casting, normalization, and 6 deterministic DQ flags.
    Uses try_to_timestamp() for ISO 8601 tolerance.
    Uses regexp_extract == '' pattern to avoid boolean cast errors on card format.
    """
    transformed = (
        df
        .withColumn("transaction_timestamp",
                    expr("try_to_timestamp(transaction_timestamp)"))
        .withColumn("amount",
                    regexp_replace(col("amount"), "[^0-9.]", "").cast(DecimalType(18, 2)))
        .withColumn("currency",         upper(trim(col("currency"))))
        .withColumn("transaction_type", upper(trim(col("transaction_type"))))
        .withColumn("status",           upper(trim(col("status"))))
        .withColumn("merchant_name",    trim(col("merchant_name")))

        # DQ flags
        .withColumn("dq_amount_invalid",
                    col("amount").isNull() | (col("amount") <= 0))
        .withColumn("dq_timestamp_invalid",
                    col("transaction_timestamp").isNull())
        .withColumn("dq_currency_invalid",
                    ~col("currency").isin(VALID_CURRENCIES))
        .withColumn("dq_txn_type_invalid",
                    ~col("transaction_type").isin(VALID_TXN_TYPES))
        .withColumn("dq_status_invalid",
                    ~col("status").isin(VALID_STATUSES))
        .withColumn("dq_card_format_invalid",
                    regexp_extract(col("card_last_four"), r"^\d{4}$", 0) == "")

        # Composite pass flag
        .withColumn("dq_passed", ~(
            col("dq_amount_invalid")    | col("dq_timestamp_invalid") |
            col("dq_currency_invalid")  | col("dq_txn_type_invalid")  |
            col("dq_status_invalid")    | col("dq_card_format_invalid")
        ))
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("silver_source_table", lit(source_table_name))
    )
    return transformed

In [0]:
# COMMAND ----------
# Apply transforms
core_silver = apply_silver_transforms(core_df, "bronze_core_banking")
card_silver = apply_silver_transforms(card_df, "bronze_card_auth")

# Write passing records to Silver
core_silver.filter(col("dq_passed")).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(SILVER_CORE)

card_silver.filter(col("dq_passed")).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(SILVER_CARD)

# Route failed records to quarantine
core_failed = core_silver.filter(~col("dq_passed")) \
    .withColumn("quarantine_reason", lit("DQ_FAILED"))
card_failed = card_silver.filter(~col("dq_passed")) \
    .select([c for c in core_silver.columns if c in card_silver.columns]) \
    .withColumn("quarantine_reason", lit("DQ_FAILED"))

quarantine_df = core_failed.unionByName(card_failed, allowMissingColumns=True)

if quarantine_df.count() > 0:
    quarantine_df.write.format("delta") \
        .mode("append") \
        .saveAsTable(QUARANTINE)

print(f"[core_banking] total={core_silver.count()} "
      f"passed={core_silver.filter(col('dq_passed')).count()} "
      f"failed={core_silver.filter(~col('dq_passed')).count()}")
print(f"[card_auth]    total={card_silver.count()} "
      f"passed={card_silver.filter(col('dq_passed')).count()} "
      f"failed={card_silver.filter(~col('dq_passed')).count()}")

In [0]:
%sql
-- COMMAND ----------
%sql
-- DQ summary by flag
SELECT 'core_banking' AS source,
    SUM(CASE WHEN dq_amount_invalid    THEN 1 ELSE 0 END) AS amt_invalid,
    SUM(CASE WHEN dq_timestamp_invalid THEN 1 ELSE 0 END) AS ts_invalid,
    SUM(CASE WHEN dq_currency_invalid  THEN 1 ELSE 0 END) AS curr_invalid,
    SUM(CASE WHEN dq_txn_type_invalid  THEN 1 ELSE 0 END) AS type_invalid,
    SUM(CASE WHEN dq_status_invalid    THEN 1 ELSE 0 END) AS status_invalid,
    SUM(CASE WHEN dq_card_format_invalid THEN 1 ELSE 0 END) AS card_invalid,
    COUNT(*) AS total_passed
FROM fintech_lakehouse_dev.transactions.silver_core_banking